In [9]:
import numpy as np
import os
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms


In [31]:
X_train = pd.read_csv('./data/train.csv')
X_test = pd.read_csv('./data/test.csv')

y_train = torch.tensor((X_train["class"]-1).values, dtype = torch.long)
y_test = torch.tensor((X_test["class"]-1).values, dtype = torch.long)

X_train = torch.tensor(X_train.drop(columns="class").values, dtype = torch.float32)
X_test = torch.tensor(X_test.drop(columns="class").values, dtype = torch.float32)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=32)
test_dataloader = DataLoader(test_dataset, batch_size=32)

In [32]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(54, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [34]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

loss_fn = torch.nn.CrossEntropyLoss()
model = NeuralNetwork().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")


Epoch 1
-------------------------------
loss: 1.968252  [   32/21087]
loss: 1.320880  [ 3232/21087]
loss: 1.203444  [ 6432/21087]
loss: 0.871510  [ 9632/21087]
loss: 0.940174  [12832/21087]
loss: 0.806529  [16032/21087]
loss: 1.142891  [19232/21087]
Test Error: 
 Accuracy: 61.8%, Avg loss: 0.900063 

Epoch 2
-------------------------------
loss: 1.036715  [   32/21087]
loss: 0.936560  [ 3232/21087]
loss: 1.083271  [ 6432/21087]
loss: 0.792598  [ 9632/21087]
loss: 0.881191  [12832/21087]
loss: 0.752312  [16032/21087]
loss: 1.048493  [19232/21087]
Test Error: 
 Accuracy: 64.3%, Avg loss: 0.826449 

Epoch 3
-------------------------------
loss: 0.927314  [   32/21087]
loss: 0.866149  [ 3232/21087]
loss: 1.011469  [ 6432/21087]
loss: 0.759060  [ 9632/21087]
loss: 0.825874  [12832/21087]
loss: 0.721999  [16032/21087]
loss: 0.990206  [19232/21087]
Test Error: 
 Accuracy: 66.5%, Avg loss: 0.781288 

Epoch 4
-------------------------------
loss: 0.857347  [   32/21087]
loss: 0.816470  [ 3232/2